In [1]:
import pandas as pd

import wrds

In [1]:
a = [1, 2, 3]
a[1] += 1
a

[1, 3, 3]

In [ ]:
filepath = "AGG_holdings_clean.csv"
ishare_df = pd.read_csv(filepath, skiprows=9, header=0)
cols_to_drop = ['Location', 'Exchange', 'Currency','Yield to Call (%)', 'Asset Class']
ishare_df = ishare_df.drop(columns=cols_to_drop)
ishare_df.groupby('Sector')['Weight (%)'].sum().sort_values(ascending=False).head(12) 

In [ ]:
db = wrds.Connection(wrds_username="hauvi")
# Get the portfolio number from CRSP's metadata for AGG
agg_query = """
    SELECT * FROM crsp.fund_names WHERE ticker = 'AGG'
    """ 
agg_wrds_df = db.raw_sql(agg_query)
agg_wrds_df.sort_values(by='chgdt', ascending=False, inplace=True) 

# Get the most recent portfolio number and query the holdings specifically
portno = int(agg_wrds_df.sort_values("chgdt")["crsp_portno"].iloc[-1]) # get the most recent AGG portfolio (june 30th) idk how we can get more recent ones but whatever
col_to_keep = [ 'security_name', 'cusip', 'crsp_company_key', 'ticker', 'coupon', 'maturity_dt', 'nbr_shares', 'market_val', 'percent_tna','security_rank']
holdings = db.raw_sql(f"""
    select {', '.join(col_to_keep)} from crsp.holdings
    where crsp_portno = {portno}
      and report_dt = '2026-06-30'
""")

In [ ]:
db.get_table(library='crsp', table='tfz_dly', obs=10)

,kytreasno,kycrspid,caldt,tdbid,tdask,tdnomprc,tdnomprc_flg,tdsourcr,tdaccint,tdretnua,tdyld,tdduratn,tdpubout,tdtotout,tdpdint,tdidxratio,tdidxratio_flg
0,200001.0,19610622.800000,1961-06-14,99.956667,99.961111,99.958889,M,R,0.0,<NA>,0.000051,8.0,<NA>,3503.0,0.0,<NA>,<NA>
1,200001.0,19610622.800000,1961-06-15,99.963056,99.966944,99.965,M,R,0.0,0.000061,0.00005,7.0,<NA>,3503.0,0.0,<NA>,<NA>
2,200001.0,19610622.800000,1961-06-16,99.965833,99.971667,99.96875,M,R,0.0,0.000038,0.000052,6.0,<NA>,3503.0,0.0,<NA>,<NA>
3,200001.0,19610622.800000,1961-06-19,99.981667,99.984583,99.983125,M,R,0.0,0.000144,0.000056,3.0,<NA>,3503.0,0.0,<NA>,<NA>
4,200001.0,19610622.800000,1961-06-20,99.9875,99.989722,99.988611,M,R,0.0,0.000055,0.000057,2.0,<NA>,3503.0,0.0,<NA>,<NA>
5,200002.0,19610623.400000,1961-06-14,99.9525,99.9575,99.955,M,R,0.0,<NA>,0.00005,9.0,<NA>,1595.0,0.0,<NA>,<NA>
6,200002.0,19610623.400000,1961-06-15,99.957778,99.962222,99.96,M,R,0.0,0.00005,0.00005,8.0,<NA>,1595.0,0.0,<NA>,<NA>
7,200002.0,19610623.400000,1961-06-16,99.963056,99.965972,99.964514,M,R,0.0,0.000045,0.000051,7.0,<NA>,1595.0,0.0,<NA>,<NA>
8,200002.0,19610623.400000,1961-06-19,99.976667,99.98,99.978333,M,R,0.0,0.000138,0.000054,4.0,<NA>,1595.0,0.0,<NA>,<NA>
9,200002.0,19610623.400000,1961-06-20,99.9825,99.984583,99.983542,M,R,0.0,0.000052,0.000055,3.0,<NA>,1595.0,0.0,<NA>,<NA>


In [ ]:
# Daily returns of the AGG ETF I assume?
# can also query from crsp.daily_nav_ret 
# monthly: monthly_returns, monthly_tna_ret_nva
query = """
    SELECT * FROM crsp.daily_returns
    where crsp_fundno = 16417
"""
db.raw_sql(query).sort_values(by='caldt', ascending=False)

,crsp_fundno,caldt,dret
5728,16417.0,2026-06-30,-0.004141
5727,16417.0,2026-06-29,0.000495
5726,16417.0,2026-06-26,0.000889
5725,16417.0,2026-06-25,0.000575
5724,16417.0,2026-06-24,0.004717
...,...,...,...
4,16417.0,2003-09-26,0.003447
3,16417.0,2003-09-25,0.001282
2,16417.0,2003-09-24,0.003563
1,16417.0,2003-09-23,0.001189


In [1]:
def cusip_check_digit(cusip8):
    total = 0
    for i, ch in enumerate(cusip8.upper()):
        if ch.isdigit():
            v = int(ch)
        elif ch.isalpha():
            v = ord(ch) - ord("A") + 10
        elif ch == "*":
            v = 36
        elif ch == "@":
            v = 37
        elif ch == "#":
            v = 38
        else:
            print(f"Invalid CUSIP character: {ch}") 
        if i % 2 == 1:     
            v *= 2
        total += v // 10 + v % 10
    return str((10 - total % 10) % 10)

def to_cusip9(cusip8):
    if cusip8 is None:
        return None
    return cusip8 + cusip_check_digit(cusip8)

## Check pipeline parquet outputs (`data/processed`)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROC = Path("../data/processed")
if not PROC.exists():
    PROC = Path("data/processed")
tables = {
    "universe": pd.read_parquet(PROC / "bond_universe.parquet"),
    "prices": pd.read_parquet(PROC / "prices_daily.parquet"),
    "yield_curve": pd.read_parquet(PROC / "yield_curve_daily.parquet"),
}
for k, t in tables.items():
    print(f"== {k}: {t.shape[0]:,} rows x {t.shape[1]} cols")
    print(t.dtypes.to_string(), "\n")


In [ ]:
# 1) AGG universe: coverage and sector breakdown
u, p, c = tables["universe"], tables["prices"], tables["yield_curve"]
print(f"AGG weight in universe: {u['agg_weight'].sum():.1%}; bonds: {len(u):,}; with prices: {u['has_prices'].sum():,}")
print(u.groupby(["agg_sector", "sector"], dropna=False).agg(
    n=("cusip", "size"), with_prices=("has_prices", "sum"), weight_pct=("agg_weight", lambda s: round(s.sum() * 100, 2))))
print("\nWeight covered by prices, by asset type:")
print(u.groupby("asset_type").apply(lambda g: g.loc[g["has_prices"], "agg_weight"].sum() / g["agg_weight"].sum(),
                                    include_groups=False).round(3))


In [ ]:
# 2) PASS / FAIL checks
from IPython.display import display
KEY = [c for c in c.columns if c.startswith("zero_")]
checks = {
    "weights sum to 1": np.isclose(u["model_weight"].sum(), 1.0),
    "unique CUSIPs": u["cusip"].is_unique,
    "one row per (cusip, date)": not p.duplicated(["cusip", "date"]).any(),
    "no missing clean prices": p["clean_price"].notna().all(),
    "all prices positive": (p["clean_price"] > 0).all(),
    "dirty >= clean": (p["dirty_price"] >= p["clean_price"] - 1e-9).all(),
    "every price CUSIP is in the universe": p["cusip"].isin(u["cusip"]).all(),
    "all 8 key rates present": len(KEY) == 8 and c[KEY].notna().all().all(),
    "curve: one row per date": c["date"].is_unique,
}
for k, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + k)


In [ ]:
# 3) Solved yield vs AGG-file YTM (tests the yield maths, not the data).
# Caveat: AGG YTM is as of holdings_asof_date, prices end earlier, so compare at each bond's last price date.
last = p.sort_values("date").groupby("cusip").tail(1).merge(
    u[["cusip", "agg_ytm_pct", "holdings_asof_date"]], on="cusip")
last["diff_bps"] = (last["ytm_solved"] * 100 - last["agg_ytm_pct"]) * 100
print("Holdings date:", u["holdings_asof_date"].max().date())
print(last.groupby("sector")["diff_bps"].describe()[["count", "mean", "50%"]].round(1))


In [ ]:
# 4) Sample window and staleness
print(f"Sample: {p['date'].min().date()} to {p['date'].max().date()}; holdings of {u['holdings_asof_date'].max().date()}")
print("Last priced date by source:", p.groupby("source")["date"].max().dt.date.to_dict())
print("Curve range:", c["date"].min().date(), "to", c["date"].max().date())
print("\nRows / bonds / median days per bond by source:")
print(p.groupby("source").agg(rows=("date", "size"), bonds=("cusip", "nunique")).assign(
    days_per_bond=lambda d: (d["rows"] / d["bonds"]).round(1)))
# stale = gap since the bond's previous price observation
gap = p.sort_values(["cusip", "date"]).groupby("cusip")["date"].diff().dt.days
print("\nShare of observations >5 days after the previous one, by source:")
print((gap > 5).groupby(p["source"]).mean().round(3))


In [ ]:
# 5) Look at the tables
display(tables["prices"].head())
display(tables["yield_curve"].tail())
print(tables["yield_curve"][KEY + ["fit_rmse_price", "n_bonds_used"]].describe().round(3))
